# AlBERTo, preprocessing + enriched hashtag/PER/ORG inputs + followers + replies
# Workflow

1. Load and inspect the data.

2. Preprocess tweets: remove URLs and mentions, normalize whitespace, handle emojis, and tag politician and party names within hashtags using Wikidata dictionaries.
- WIKI dictionary: obtained through the SPARQL endpoint. It find matchs in the tweets texts and add the tags <entity> word </entity>  around only match word when there is a hashtag such as #salvinimeloni or #salvini. This also includes a pre-processing of hastags.

3. Check conflicting labels and remove duplicate preprocessed texts. Create a stratified 90% training / 10% validation split using seed 42.

 The preprocessing also includes a hastag special treatment for extracting entities in the following phase.

4. Prepare the model and features:
   - Load Alberto model `  "m-polignano-uniba bert_uncased_L-12_H-768_A-12_italian_alb3rt0"` and its tokenizer.
   - Add annotation markers and frequent unsupported emojis identified only in training data; resize embeddings.
   - Tokenize with a maximum length of 256 tokens and dynamic padding.
   - Pre-process follower count and the reply on train data using imputation (median and mode) and scaling.

5. Fine-tune the preprocessed tweet text + features at a learning rate of 5e-6. Evaluate every 100 steps with early stopping. Select the checkpoint with the best validation macro-F1.

6. Evaluate test accuracy, macro-F1, HATE precision/recall/F1, and the confusion matrix. Export results.

Libraries import and installs

In [ ]:
pip install emoji

In [1]:
import pandas as pd
import re
import collections
import gc
from collections import Counter
import unicodedata

In [2]:
print(re.match('chat', 'chat noir'))

<re.Match object; span=(0, 4), match='chat'>


In [3]:
print(re.match('chat', 'le chat noir'))

None


In [ ]:
import html
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers import AddedToken
from transformers import EarlyStoppingCallback

from torch import nn
from transformers import BertModel, BertPreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import data

In [ ]:
training_textual_path = '/content/drive/MyDrive/master_unipa/training_textual.csv'
training_contextual_path = '/content/drive/MyDrive/master_unipa/training_contextual.csv'

test_textual_path = '/content/drive/MyDrive/master_unipa/test_textual_gold.csv'
test_contextual = '/content/drive/MyDrive/master_unipa/test_contextual.csv'

In [ ]:
df_train = pd.read_csv(training_textual_path)
df_train_contextual = pd.read_csv(training_contextual_path)

df_test = pd.read_csv(test_textual_path)
df_test_contextual = pd.read_csv(test_contextual)

In [ ]:
df_train.head(2)
display(df_train.head(3))
display(df_train.shape)
display(df_train.label.value_counts())
display(df_train.dataset.value_counts())

print(f'Are tweets ids unique? {df_train["anonymized_tweet_id"].is_unique}')
print(f'Are there any repeated lines? {df_train.duplicated().any()}')


,anonymized_tweet_id,anonymized_text,label,dataset
0,217874450618134,@unknown Con tutte le denunce che si sta becca...,1,training_politics
1,360042217507605,#PRESCRIZIONE : I (t)re magi hanno trovato l’a...,0,training_politics
2,817917484817935,Il #m5s ha votato contro le #unionicivili addu...,1,training_politics


(5600, 4)

,count
label,
0,3456
1,2144


,count
dataset,
training_politics,5600


Are tweets ids unique? True
Are there any repeated lines? False


In [ ]:
print(df_test["dataset"].value_counts())

df_train = df_train.loc[
    df_train["dataset"].eq("training_politics")
].copy()

df_test = df_test.loc[
    df_test["dataset"].eq("test_politics")
].copy()

assert len(df_train) == 5600
assert len(df_test) == 1400

dataset
test_religious    3000
test_politics     1400
Name: count, dtype: int64


In [ ]:
df_test["dataset"].value_counts()

,count
dataset,
test_politics,1400


In [ ]:
df_test.head(2)

,anonymized_tweet_id,anonymized_text,label,dataset
0,424801448454884,questo anno #babbonatale portato giocattolo it...,1,test_politics
1,775194088981616,Adesso che in mezzo alla strada grazie al verg...,1,test_politics


In [ ]:
df_train["anonymized_tweet_id"].is_unique

True

In [ ]:
df_train['anonymized_tweet_id'].duplicated().any()

np.False_

In [ ]:
df_train_contextual.head(2)
display(df_train_contextual.head(3))
display(df_train_contextual.shape)
display(df_train_contextual.columns)
display(df_train.dataset.value_counts())
assert (df_train.anonymized_tweet_id == df_train_contextual.anonymized_tweet_id).all()

print(f'Are tweets ids unique? {df_train_contextual["anonymized_tweet_id"].is_unique}')
print(f'Are there any repeated lines? {df_train_contextual.duplicated().any()}')

,anonymized_tweet_id,created_at,retweet_count,favorite_count,source,is_reply,is_retweet,is_quote,anonymized_user_id,user_created_at,statuses_count,followers_count,friends_count,anonymized_description,dataset
0,217874450618134,2018-08-11,0.0,6.0,"<a href=""http://twitter.com/download/iphone"" r...",1.0,0.0,0.0,8.868893e+14,2018-04-01,554.0,748.0,753.0,Avete presente quegli stereotipi sui siciliani...,training_politics
1,360042217507605,2018-11-08,154.0,448.0,"<a href=""http://twitter.com/download/iphone"" r...",0.0,0.0,0.0,2.570741e+14,2011-10-26,35043.0,10838.0,4535.0,Prof di latino e Deputata PD.Ama lo sport la R...,training_politics
2,817917484817935,2018-11-10,195.0,638.0,"<a href=""http://twitter.com/download/iphone"" r...",0.0,0.0,0.0,6.908473e+14,2009-09-24,139750.0,7971.0,1866.0,"Medico, appassionato di Cinema d'Autore, Music...",training_politics


(5600, 15)

Index(['anonymized_tweet_id', 'created_at', 'retweet_count', 'favorite_count',
       'source', 'is_reply', 'is_retweet', 'is_quote', 'anonymized_user_id',
       'user_created_at', 'statuses_count', 'followers_count', 'friends_count',
       'anonymized_description', 'dataset'],
      dtype='object')

,count
dataset,
training_politics,5600


Are tweets ids unique? True
Are there any repeated lines? False


In [ ]:

politicians_df = pd.read_csv(
    "/content/drive/MyDrive/master_unipa/politicians.csv",
    dtype="string",
    encoding="utf-8-sig",
)

parties_df = pd.read_csv(
    "/content/drive/MyDrive/master_unipa/parties.csv",
    dtype="string",
    encoding="utf-8-sig",
)

Wiki data

In [ ]:
def compact_key(text):
    """ cercare una voce dentro un hashtag."""
    text = unicodedata.normalize("NFC", text).casefold()
    return re.sub(r"[\s_-]+", "", text)


def collect_entries(frame, columns):
    entries = {}

    for column in columns:
        if column not in frame.columns:
            continue

        for value in frame[column].dropna():
            value = str(value).strip()

            # Esclude valori vuoti e identificatori senza etichetta.
            if not value or re.fullmatch(r"Q\d+", value):
                continue

            key = compact_key(value)
            entries.setdefault(key, set()).add(value)

    return entries


politician_entries = collect_entries(
    politicians_df,
    ["nome", "cognome", "alias"],
)

party_entries = collect_entries(
    parties_df,
    ["partitoLabel", "alias", "sigla"],
)

political_entries = {}

for source in [politician_entries, party_entries]:
    for key, variants in source.items():
        political_entries.setdefault(key, set()).update(variants)



Pre processing

In [ ]:
print("Politician columns:", politicians_df.columns.tolist())
print("Party columns:", parties_df.columns.tolist())

print("Politician entries:", len(politician_entries))
print("Party entries:", len(party_entries))

assert politician_entries, "Politician dictionary is empty."
assert party_entries, "Party dictionary is empty."

for key in ["salvini", "meloni", "m5s", "pd"]:
    print(key, "→", political_entries.get(key, "NOT FOUND"))

Politician columns: ['persona', 'nome', 'cognome']
Party columns: ['partito', 'partitoLabel', 'alias', 'sigla']
Politician entries: 4923
Party entries: 1120
salvini → {'Salvini'}
meloni → {'Meloni'}
m5s → {'M5S'}
pd → {'PD'}


In [ ]:
import re
import regex
import emoji


class CustomPreprocessor:
    pattern = re.compile(
        r"(?P<url>(?:https?://|www\.)[^\s<>]+)"
        r"|(?P<user>(?<![\w@])@[A-Za-z0-9_]+)"
        r"|(?P<hashtag>(?<![\w#])#(?:(?!ー)\w)+)",
        flags=re.IGNORECASE,
    )

    COVID_PATTERN = re.compile(
        r"\bcovid(?P<suffix>[\s_\-‐‑‒–—―−﹘﹣－＿ー]*19)?\b",
        flags=re.IGNORECASE,
    )

    def preprocess(self, text):
        if not isinstance(text, str):
            raise TypeError("Fill missing text before preprocessing.")

        def replace(match):
            if match.lastgroup == "url":
                url = match.group()
                trimmed = url.rstrip(".,!?;:")
                return " " + url[len(trimmed):]

            if match.lastgroup == "user":
                return " "

            return f"<hashtag> {match.group()[1:]} </hashtag>"

        # Remove existing URL and [user]
        text = re.sub(
            r"\[url\]|\[user\]|<url>|<user>",
            " ",
            text,
            flags=re.IGNORECASE,
        )

        text = text.translate(str.maketrans({
            "“": '"',
            "”": '"',
            "‘": "'",
            "’": "'",
        }))

        text = self.pattern.sub(replace, text)

        text = self.COVID_PATTERN.sub(
            lambda match: "covid" + (match.group("suffix") or ""),
            text,
        )

        text = re.sub(
            r"([_\-‐‑‒–—―−﹘﹣－＿ー])",
            r" \1 ",
            text,
        )

        replacements = {
            "❤️": " cuore ",
            "❤": " cuore ",
            ":)": " sorriso ",
            ":(": " tristezza ",
        }

        for symbol, replacement in replacements.items():
            text = text.replace(symbol, replacement)

        text = emoji.replace_emoji(
            text,
            replace=lambda chars, data: f" {chars} ",
        )

        return re.sub(r"\s+", " ", text).strip()


prepare_text = CustomPreprocessor()

Entities matching and preprocessing

In [ ]:

SHORT_PARTY_ALIASES = {"m5s", "pd", "fdi", "lega", "leu"}
EXCLUDE_FROM_SUBSTRING = {"aprile", "buffoni", "rossi"}


person_keys = {
    key for key in politician_entries
    if key.isalnum() and len(key) >= 4
}

party_keys = {
    key for key in party_entries
    if key.isalnum()
    and (len(key) >= 4 or key in SHORT_PARTY_ALIASES)
}

ambiguous_keys = person_keys & party_keys

entity_types = {
    key: "per" for key in person_keys - ambiguous_keys
}
entity_types.update({
    key: "org" for key in party_keys - ambiguous_keys
})


substring_keys = sorted(
    (
        key for key in entity_types
        if (len(key) >= 4 or key == "m5s")
        and key not in EXCLUDE_FROM_SUBSTRING
    ),
    key=lambda key: (-len(key), key),
)




if not substring_keys:
    raise ValueError("No substring keys.")

political_pattern = re.compile(
    "|".join(re.escape(key) for key in substring_keys),
    flags=re.IGNORECASE,
)


def separate_and_tag_political_names(text):
    def process_hashtag(match):
        word = match.group(1).strip()

        # First check the entire hashtag
        key = word.casefold()
        if key in entity_types:
            tag = entity_types[key]
            return f"<hashtag> <{tag}> {word} </{tag}> </hashtag>"

        pieces = []
        cursor = 0

        # find names inside the hashtag.
        for entity in political_pattern.finditer(word):
            pieces.append(word[cursor:entity.start()])

            matched_text = entity.group()
            tag = entity_types[matched_text.casefold()]
            pieces.append(f"<{tag}> {matched_text} </{tag}>")

            cursor = entity.end()

        pieces.append(word[cursor:])
        content = " ".join(piece for piece in pieces if piece)

        return f"<hashtag> {content} </hashtag>"

    return re.sub(
        r"<hashtag>\s*([^<>]*?)\s*</hashtag>",
        process_hashtag,
        text,
    )




def split_numbers_outside_entities(text):
    pattern = regex.compile(
        r"<per>.*?</per>(*SKIP)(*F)"
        r"|<org>.*?</org>(*SKIP)(*F)"
        r"|(?<=\p{L})(?=\p{Nd})"
        r"|(?<=\p{Nd})(?=\p{L})",
        flags=regex.DOTALL,
    )
    return pattern.sub(" ", text)


def preprocess_text(text):
    cleaned = prepare_text.preprocess(text)
    tagged = separate_and_tag_political_names(cleaned)
    return split_numbers_outside_entities(tagged)



## Preprocessing and duplicates drop

In [ ]:
df = df_train.copy()

if df["anonymized_text"].isna().any():
    raise ValueError("Missing tweet text.")

df["anonymized_text"] = df["anonymized_text"].map(preprocess_text)

if df["anonymized_text"].eq("").any():
    raise ValueError("Empty text after preprocessing.")

if df.groupby("anonymized_text")["label"].nunique().gt(1).any():
    #raise ValueError("Some preprocessed texts have conflicting labels.")

    df = df.drop_duplicates(subset="anonymized_text").reset_index(drop=True)


if df_test["anonymized_text"].isna().any():
    raise ValueError("Missing test tweet text.")

df_test["anonymized_text"] = df_test["anonymized_text"].map(preprocess_text)

if df_test["anonymized_text"].eq("").any():
    raise ValueError("Empty test text after preprocessing.")

df_test.to_csv("test.csv", index=False)


## Load split function

In [ ]:
def load_split(path):
    df = pd.read_csv(path)

    required = {"anonymized_text", "label", "anonymized_tweet_id"}
    if not required.issubset(df.columns):
        raise ValueError(f"{path}: missing required columns.")

    if df[["anonymized_text", "label"]].isna().any().any():
        raise ValueError(f"{path}: missing text or labels.")

    labels = pd.to_numeric(df["label"], errors="raise")
    if not labels.isin([0, 1]).all():
        raise ValueError(f"{path}: labels must be 0 or 1.")

    text = df["anonymized_text"].astype(str).str.strip()
    if text.eq("").any():
        raise ValueError(f"{path}: empty text.")

    result = pd.DataFrame({
        "anonymized_tweet_id": df["anonymized_tweet_id"],
        "text": text,
        "labels": labels.astype(int),
    }).reset_index(drop=True)

    print(f"\n{path}: {len(result)} rows")
    print(result["labels"].value_counts().sort_index())

    return result

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    stratify=df["label"],  # Preserve class proportions
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)

print(f"Train: {len(train_df)} | Validation: {len(val_df)}")
print("Train proportions:\n", train_df["label"].value_counts(normalize=True))
print("Validation proportions:\n", val_df["label"].value_counts(normalize=True))

Train: 5026 | Validation: 559
Train proportions:
 label
0    0.61739
1    0.38261
Name: proportion, dtype: float64
Validation proportions:
 label
0    0.617174
1    0.382826
Name: proportion, dtype: float64


In [ ]:
train_df = load_split("train.csv")
val_df = load_split("validation.csv")
test_df = load_split('test.csv')



train.csv: 5026 rows
labels
0    3103
1    1923
Name: count, dtype: int64

validation.csv: 559 rows
labels
0    345
1    214
Name: count, dtype: int64

test.csv: 1400 rows
labels
0    700
1    700
Name: count, dtype: int64


### Contextual features: join

In [ ]:
from sklearn.preprocessing import StandardScaler

# contextual inf joined by tweet ID
def attach_context(frame, contextual):
    columns = ['anonymized_tweet_id', 'followers_count', 'is_reply']
    if contextual.duplicated('anonymized_tweet_id').any():
        raise ValueError('Duplicate contextual tweet IDs: inspect before joining.')
    # Allow rerunning this cell without creating _x/_y metadata columns.

    frame = frame.drop(columns=columns[1:], errors='ignore')
    merged = frame.merge(contextual[columns], on='anonymized_tweet_id',
                         how='left', validate='many_to_one', sort=False)

    for column in columns[1:]:
        merged[column] = pd.to_numeric(merged[column], errors='raise')

    if (merged['followers_count'].dropna() < 0).any():
        raise ValueError('followers_count must be nonnegative.')

    if not merged['is_reply'].dropna().isin([0, 1]).all():
        raise ValueError('is_reply must be 0/1.')

    print('Missing contextual values:', merged[columns[1:]].isna().sum().to_dict())
    return merged

train_df = attach_context(train_df, df_train_contextual)
val_df = attach_context(val_df, df_train_contextual)
test_df = attach_context(test_df, df_test_contextual)

# Imputation and scaling from the training set.
followers_fill = train_df['followers_count'].median()

reply_modes = train_df['is_reply'].mode()

if pd.isna(followers_fill) or reply_modes.empty:
    raise ValueError('No observed training metadata for one of the features.')
reply_fill = float(reply_modes.iloc[0])

def followers_log(frame):
    return np.log1p(frame['followers_count'].fillna(followers_fill).to_numpy(dtype=float)).reshape(-1, 1)
followers_scaler = StandardScaler().fit(followers_log(train_df))

for frame in (train_df, val_df, test_df):
    frame['followers_feature'] = followers_scaler.transform(followers_log(frame)).ravel().astype('float32')

    # Reply is a binary feature: no standardization needed.
    frame['reply_feature'] = frame['is_reply'].fillna(reply_fill).astype('float32')


Missing contextual values: {'followers_count': 73, 'is_reply': 73}
Missing contextual values: {'followers_count': 5, 'is_reply': 5}
Missing contextual values: {'followers_count': 0, 'is_reply': 0}


# Model and tokenizer


In [ ]:
MODEL_NAME = (
    "m-polignano-uniba/"
    "bert_uncased_L-12_H-768_A-12_italian_alb3rt0"
)

SEED = 42
MAX_LENGTH = 256  # Tokens, not characters
OUTPUT_DIR = Path("outputs/alberto_clean_wikidata") # output directory


set_seed(SEED)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

In [ ]:
class AlbertoWithContext(BertPreTrainedModel):
    def __init__(self, config):

        super().__init__(config)

        self.bert = BertModel(config)

        dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        self.dropout = nn.Dropout(dropout)

        # Pooled tweet embedding + followers + reply flag: 768 + 2 -> 2.
        self.classifier = nn.Linear(config.hidden_size + 2, config.num_labels)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                followers_feature=None, reply_feature=None, labels=None):

        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids, return_dict=True)
        text_vector = self.dropout(outputs.pooler_output)

        if followers_feature is None or reply_feature is None:
            raise ValueError('Both contextual features are required.')

        context = torch.stack([followers_feature.reshape(-1), reply_feature.reshape(-1)], dim=-1)

        context = context.to(device=text_vector.device, dtype=text_vector.dtype)

        logits = self.classifier(torch.cat([text_vector, context], dim=-1))

        loss = None if labels is None else nn.functional.cross_entropy(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

model = AlbertoWithContext.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    classifier_dropout=0.2,
    id2label={0: "NON_HATE", 1: "HATE"},
    label2id={"NON_HATE": 0, "HATE": 1},
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  740MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] AlbertoWithContext LOAD REPORT from: m-polignano-uniba/bert_uncased_L-12_H-768_A-12_italian_alb3rt0
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because m

## Add emojis found in train to the tokenizer

In [ ]:
emoji_counts = Counter()

text_columns = ["text"]
if "description" in train_df.columns:
    text_columns.append("description")

for column in text_columns:
    for text in train_df[column].fillna(""):
        for match in emoji.emoji_list(text):
            emoji_counts[match["emoji"]] += 1

In [ ]:
MIN_EMOJI_COUNT = 2

def unsupported(symbol):
    ids = tokenizer.encode(symbol, add_special_tokens=False)
    return not ids or tokenizer.unk_token_id in ids

emoji_tokens = [
    symbol for symbol, count in emoji_counts.items()
    if count >= MIN_EMOJI_COUNT and unsupported(symbol)
]


training_texts = train_df["text"].fillna("")

# added tokens: each annotation becomes one token
markers = ['<per>', '</per>', '<org>', '</org>', '<hashtag>', '</hashtag>']


other_tokens = ['covid']

candidates = list(dict.fromkeys(emoji_tokens + other_tokens + markers ))


num_added = tokenizer.add_tokens([
    AddedToken(
        token,
        normalized=(token in other_tokens),
        single_word=(token in other_tokens),
        lstrip=False,
        rstrip=False,
    )
    for token in candidates
])


if num_added:
    model.resize_token_embeddings(len(tokenizer))

print("New vocabulary entries:", num_added)
print("Emoji candidates:", emoji_tokens)

for token in candidates:
    assert tokenizer.tokenize(token) == [token], (token, tokenizer.tokenize(token))
assert model.get_input_embeddings().num_embeddings == len(tokenizer)


[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


New vocabulary entries: 109
Emoji candidates: ['🤦🏻\u200d♂️', '🤐', '‼️', '👎', '😡', '🙄', '⚠️', '🍒', '🤮', '🛑', '🔴', '🤔', '😲', '🇮🇹', '📣', '🤣', '🧐', '🤬', '☠️', '🤡', '🥎', '🪴', '😀', '🖕🏻', '👉', '😅', '🗽', '😳', '😷', '👇', '💩', '😎', '😶', '❌', '✅', '🤦\u200d♂️', '✔', '👌', '➡️', '🤷\u200d♂️', '🌾', '🌹', '🐬', '🤚', '🖤', '🇱🇷', '🏴\u200d☠️', '😑', '👇🏼', '🎶', '🤢', '👏🏻', '💔', '💪', '🍾', '🔵', '🤭', '🙉', '🙊', '✊', '🔥', '🇨🇳', '🖕', '📲', '➡', '🤗', '🇪🇺', '💥', '😬', '🍺', '⁉️', '❗', '▶️', '🤦\u200d♀️', '⚡', '🎪', '🤫', '🤦🏻\u200d♀️', '😤', '🤧', '💤', '🙂', '⬇️', '💸', '🐲', '🤷🏻\u200d♂️', '🦄', '🤞🏻', '⤵️', '🔸', '😆', '🤑', '👆', '👋', '👎🏻', '✔️', '🖐', '❓', '😒', '🔽', '✌️', '🤞']


In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

### make dataset

In [ ]:
def make_dataset(frame):
    dataset = Dataset.from_pandas(frame[['text', 'labels', 'followers_feature', 'reply_feature']], preserve_index=False)
    return dataset.map(
        tokenize,
        batched=True,
        remove_columns=["text"],
    )


train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)
test_dataset = make_dataset(test_df)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/5026 [00:00<?, ? examples/s]

Map:   0%|          | 0/559 [00:00<?, ? examples/s]

Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

In [ ]:
for text in [
    "Ciao @mario!",
    "Leggi [URL] e commenta.",
    "Guarda https://example.com!",
    "#salvinibugiardo @123 [URL]",
]:
    print(repr(text), "→", repr(preprocess_text(text)))

'Ciao @mario!' → 'Ciao !'
'Leggi [URL] e commenta.' → 'Leggi e commenta.'
'Guarda https://example.com!' → 'Guarda !'
'#salvinibugiardo @123 [URL]' → '<hashtag> <per> salvini </per> bugiardo </hashtag>'


## Trainer and evaluation metrics

- best checkpoint is selected using validation macro-F1

In [ ]:
# Evaluation metrics
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            labels=[0, 1],
            average="macro",
            zero_division=0,
        ),
        "hate_precision": precision,
        "hate_recall": recall,
        "hate_f1": f1,
    }


# Fine-tuning configuration
training_args = TrainingArguments(
    learning_rate=5e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,

    weight_decay=0.01,
    optim="adamw_torch",

    # chosen eval strategy for check the modelling before epoch completition since we have a small amount of training data
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",  # best checkpoint selection
    greater_is_better=True,
    save_total_limit=1,  #disk space clean
    logging_steps=50,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=4,
            early_stopping_threshold=0.0,
        )]
)


# Train
trainer.train()


Step,Training Loss,Validation Loss,Accuracy,Macro F1,Hate Precision,Hate Recall,Hate F1
100,0.447753,0.314917,0.874776,0.867021,0.842857,0.827103,0.834906
200,0.273750,0.244541,0.906977,0.901032,0.889423,0.864486,0.876777
300,0.273227,0.238982,0.915921,0.911716,0.874439,0.911215,0.892449
400,0.266144,0.236760,0.912343,0.906829,0.894737,0.873832,0.884161
500,0.220292,0.292771,0.883721,0.880484,0.794466,0.939252,0.860814
600,0.237424,0.249321,0.914132,0.910198,0.864035,0.920561,0.891403
700,0.185735,0.255458,0.919499,0.914896,0.893023,0.897196,0.895105
800,0.205096,0.250340,0.910555,0.905357,0.883178,0.883178,0.883178
900,0.213587,0.287872,0.914132,0.910198,0.864035,0.920561,0.891403
1000,0.159981,0.276962,0.919499,0.915610,0.875556,0.920561,0.897494


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1400, training_loss=0.2249075562613351, metrics={'train_runtime': 97.2211, 'train_samples_per_second': 229.763, 'train_steps_per_second': 14.4, 'total_flos': 935891206511976.0, 'train_loss': 0.2249075562613351, 'epoch': 4.444444444444445})

## Prediction on test set and evaluation

In [ ]:
# test evaluation
test_output = trainer.predict(test_dataset)
test_predictions = np.argmax(test_output.predictions, axis=-1)

print("\nTest metrics:")
print(test_output.metrics)

print("\nClassification report:")
print(classification_report(
    test_df["labels"],
    test_predictions,
    labels=[0, 1],
    target_names=["NON_HATE", "HATE"],
    digits=4,
    zero_division=0,
))

print("\nConfusion matrix: rows=true labels, columns=predictions")
print(confusion_matrix(
    test_df["labels"],
    test_predictions,
    labels=[0, 1],
))




results = test_df.rename(columns={"labels": "true_label"}).copy()
results["predicted_label"] = test_predictions
results["probability_hate"] = torch.softmax(
    torch.tensor(test_output.predictions),
    dim=-1,
)[:, 1].numpy()


Test metrics:
{'test_loss': 0.29468095302581787, 'test_accuracy': 0.915, 'test_macro_f1': 0.9149728867623604, 'test_hate_precision': 0.9303703703703704, 'test_hate_recall': 0.8971428571428571, 'test_hate_f1': 0.9134545454545454, 'test_runtime': 0.5708, 'test_samples_per_second': 2452.695, 'test_steps_per_second': 77.085}

Classification report:
              precision    recall  f1-score   support

    NON_HATE     0.9007    0.9329    0.9165       700
        HATE     0.9304    0.8971    0.9135       700

    accuracy                         0.9150      1400
   macro avg     0.9155    0.9150    0.9150      1400
weighted avg     0.9155    0.9150    0.9150      1400


Confusion matrix: rows=true labels, columns=predictions
[[653  47]
 [ 72 628]]


### Print best checkpoint results

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)

Best checkpoint: trainer_output/checkpoint-1000
Best validation macro-F1: 0.9156101864929331


In [ ]:
print("Classifier:", model.config.classifier_dropout)
print("Hidden:", model.config.hidden_dropout_prob)
print("Attention:", model.config.attention_probs_dropout_prob)

Classifier: 0.2
Hidden: 0.1
Attention: 0.1


## Error analysis

### Save the predictions on val and test sets for doing an error analysis

In [ ]:

ERROR_DIR = Path(
    "/content/drive/MyDrive/master_unipa/alberto_wiki_error_analysis_final"
)
ERROR_DIR.mkdir(parents=True, exist_ok=True)


def save_error_analysis(trainer, dataset, frame, split_name):

    output = trainer.predict(dataset, metric_key_prefix=split_name)

    logits = output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = torch.softmax(
        torch.tensor(logits), dim=-1
    )[:, 1].numpy()

    assert len(frame) == len(logits)
    np.testing.assert_array_equal(
        frame["labels"].to_numpy(),
        output.label_ids,
    )

    analysis = frame.copy().reset_index(drop=True)
    analysis["prediction"] = logits.argmax(axis=-1)
    analysis["probability_hate"] = probabilities
    analysis["correct"] = analysis["labels"].eq(analysis["prediction"])

    false_negatives = analysis[
        (analysis["labels"] == 1) & (analysis["prediction"] == 0)
    ].sort_values("probability_hate")

    false_positives = analysis[
        (analysis["labels"] == 0) & (analysis["prediction"] == 1)
    ].sort_values("probability_hate", ascending=False)

    for suffix, result in [
        ("predictions", analysis),
        ("false_negatives", false_negatives),
        ("false_positives", false_positives),
    ]:
        result.to_csv(
            ERROR_DIR / f"{split_name}_{suffix}.csv",
            index=False,
            encoding="utf-8-sig",
        )

    print(f"\n{split_name.upper()}")
    print(f"False negatives: {len(false_negatives)}")
    print(f"False positives: {len(false_positives)}")

    with pd.option_context("display.max_colwidth", None):
        print("\nFalse negatives — most confident mistakes first:")
        display(false_negatives.head(20))

        print("\nFalse positives — most confident mistakes first:")
        display(false_positives.head(20))

    return analysis, false_negatives, false_positives


val_analysis, val_fn, val_fp = save_error_analysis(
    trainer, val_dataset, val_df, "validation"
)

test_analysis, test_fn, test_fp = save_error_analysis(
    trainer, test_dataset, test_df, "test"
)

print(f"\nSaved CSV files to: {ERROR_DIR}")


VALIDATION
False negatives: 17
False positives: 28

False negatives — most confident mistakes first:


,anonymized_tweet_id,text,labels,followers_count,is_reply,followers_feature,reply_feature,prediction,probability_hate,correct
352,102713507179810,"In <hashtag> senato </hashtag> si sta votando ORA una vera e propria SANATORIA dei MISFATTI del <hashtag> Governo </hashtag> <hashtag> <per> Conte </per> </hashtag>. Chiedono la conversione in <hashtag> Legge </hashtag> del <hashtag> decreto </hashtag> del <hashtag> 25 marzo </hashtag>. In parole povere, VOGLIONO PARARSI IL SEDERE SAPENDO D'AVER AGITO SENZA ALCUNA LEGITTIMAZIONE. <hashtag> 20 maggio </hashtag>",1,7127.0,0.0,0.924851,0.0,0,0.004363,False
346,757713758241511,"Su coraggio, e quanto prima diffondete la lista dei partecipanti (anche se sarà lunga e da dividere zona per zona)! Così sapremo quali saranno i locali da frequentare anche dopo che ci saremo liberati dall'imbecille che si spaccia per <hashtag> ministro </hashtag>! <hashtag> ioapro </hashtag> <hashtag> voiaprite </hashtag> <hashtag> noiveniamo </hashtag>",1,436.0,1.0,-0.400817,1.0,0,0.004551,False
406,733879466972060,"Sono iscritto al <hashtag> <org> M5S </org> </hashtag> dal 2010 e voterò NO per l'autorizzazione a procedere, non voglio più vedere nuovamente l'italia in mano al PD, Berlusconi, e altre bande di criminali <hashtag> Diciotti </hashtag> <hashtag> Saviano </hashtag> <hashtag> 17 Febbraio </hashtag> <hashtag> ottoemezzo </hashtag> <hashtag> mezzorainpiu </hashtag> <hashtag> <org> m5s </org> </hashtag> <hashtag> BuonaDomenica </hashtag> <hashtag> nonelarena </hashtag> <hashtag> Stasera <per> Italia </per> </hashtag>",1,3065.0,0.0,0.524254,0.0,0,0.006428,False
449,12161801310064,Sarò sempre grato al <hashtag> <org> m5s </org> </hashtag>. Far sparire <hashtag> <per> dibattista </per> </hashtag> è un'opera di grande valore civico.,1,40062.0,0.0,1.744617,0.0,0,0.006718,False
264,174560026342718,"Se non riparte tutto non serve, cacciamo questo <hashtag> governo </hashtag>, non si vince in modo pacifico e inevitabile che ci siano vittime ma è meglio morire lottando che subire questa dittatura. Prendiamo i loro <hashtag> stipendi </hashtag> i loro beni i loro <hashtag> <per> vitali </per> zi </hashtag> e facciamo <hashtag> ripartire </hashtag> le attività.",1,441.0,1.0,-0.395415,1.0,0,0.009089,False
526,48292635441696,Da quando la città di Livorno non è piu amministrata da <hashtag> <per> Nogarin </per> </hashtag> e dal <hashtag> <org> m5s </org> </hashtag> non si sente più nulla. Tutto funziona bene e nessun pennivendolo attacca il nuovo sindaco. ECCHECASO. <hashtag> <per> raggi </per> </hashtag> <hashtag> roma </hashtag> <hashtag> SeaWacht 3 </hashtag> <hashtag> Olimpiade 2026 </hashtag>,1,369.0,0.0,-0.479844,0.0,0,0.010183,False
475,628289366412868,"Si, avrebbe potuto accadere altrove. Ma la prima regione infettata dell'Italia è la regione Lombardia sotto il governo <hashtag> <per> salvini </per> virus </hashtag>. E le altre no. Questo è un fatto. <hashtag> coronavirus <per> italIa </per> </hashtag>",1,1478.0,1.0,0.178096,1.0,0,0.010232,False
155,921041004502023,"cacciamo questo <hashtag> governo </hashtag> compreso Mattarella, non si vince in modo pacifico e inevitabile che ci siano vittime ma è meglio morire lottando che subire questa dittatura. Prendiamo i loro <hashtag> stipendi </hashtag> i loro beni i loro <hashtag> <per> vitali </per> zi </hashtag> e facciamo <hashtag> ripartire </hashtag> le attività.",1,441.0,0.0,-0.395415,0.0,0,0.012066,False
438,930674284455273,Ma La Palombella e stupida chiede <hashtag> <per> Fontana </per> </hashtag> se per le riaperture e con <hashtag> <per> Salvini </per> </hashtag> o <hashtag> <per> Letta </per> </hashtag> <hashtag> stasera <per> italia </per> </hashtag>,1,573.0,0.0,-0.271331,0.0,0,0.012662,False
556,525953232791089,"<hashtag> <per> renzi </per> </hashtag>, il tuo socio <hashtag> <per> Gozi </per> </hashtag> avrà stappato una magnum 🥂 🍾 🥂 , tra poco, il primo trombato di <hashtag> Macron </hashtag>, potrà scaldare uno scranno a <hashtag> Bruxelles </hashtag>


False positives — most confident mistakes first:


,anonymized_tweet_id,text,labels,followers_count,is_reply,followers_feature,reply_feature,prediction,probability_hate,correct
271,717993700216178,<hashtag> <per> Conte </per> </hashtag> <hashtag> sivergogni </hashtag> e aggiungo <hashtag> <per> Conte </per> Dimettiti </hashtag> <hashtag> <per> conte </per> vaia <per> casa </per> </hashtag> per il bene dell'Italia e degli Italiani.,0,31.0,0.0,-1.642128,0.0,1,0.994302,False
18,790834966216958,"Quel partito sembra aver recepito la lezione di un buffone del foro... quel tale che svelava a tutti come, per fare <hashtag> politica </hashtag>, servano ""anima e cacciavite""!",0,8404.0,1.0,1.003102,1.0,1,0.994256,False
537,735405138734024,Quindi scopriamo oggi che un tentato reato non è reato. Ma chi credete di pigliare per il culo? <hashtag> Savoini </hashtag> <hashtag> buzzfeed </hashtag> <hashtag> <per> salvini </per> </hashtag>,0,78841.0,0.0,2.066076,0.0,1,0.993318,False
432,641974717504669,"Anche San Marino, lì non hanno <hashtag> <per> speranza </per> </hashtag>, ma <hashtag> certezza </hashtag>. <hashtag> <per> speranza </per> dimettiti </hashtag> <hashtag> TachipirinaEVigileAttesa </hashtag>. <hashtag> SantaInquisizioneCiFaUnaPippa </hashtag>",0,791.0,1.0,-0.118467,1.0,1,0.992332,False
508,101150690791556,"<hashtag> <per> Tommasi </per> </hashtag> proclama lo sciopero dei <hashtag> calciatori </hashtag> E quelli che stanno inseguendo un pallone in <hashtag> MilanGenoa </hashtag> <hashtag> parmaspal </hashtag> e <hashtag> SampVerona </hashtag> cosa sono,idraulici? Imbianchini? Ps. Non usate più il termine <hashtag> <per> paglia </per> cci </hashtag> perché a breve l'associazione pagliacci potrebbe querelarvi per diffamazione.",0,4735.0,0.0,0.730721,0.0,1,0.987290,False
35,49895334680204,"FORZA SIMO! Le figure di merda, quelle belle <hashtag> <per> pillon </per> </hashtag>",0,1059.0,0.0,0.019930,0.0,1,0.985415,False
108,826767637073989,"Salvini che ""screma"" i commenti negativi bloccando...significa che c'è un calo ed inesorabile calo dei consensi... Tutto questo è bellissimo. <hashtag> <per> salvini </per> censura <per> italia </per> ni </hashtag> <hashtag> <per> salvini </per> blocca </hashtag> <hashtag> <per> salvini </per> seiallafrutta </hashtag> <hashtag> <per> salvini </per> vergognati </hashtag> <hashtag> <per> salvini </per> cheraccontabufale </hashtag>",0,706.0,0.0,-0.172375,0.0,1,0.981103,False
58,930857600978886,Senza vergogna. <hashtag> <per> Formigoni </per> </hashtag> <hashtag> <per> vitali </per> zi </hashtag> <hashtag> <org> ForzaItalia </org> </hashtag> <hashtag> regionelombardia </hashtag> <hashtag> <per> Fontana </per> </hashtag> <hashtag> <per> fontana </per> vattene </hashtag> <hashtag> <per> sani </per> tà </hashtag> <hashtag> Salute </hashtag> <hashtag> ASL </hashtag> <hashtag> giustizia </hashtag> <hashtag> sentenza </hashtag> <hashtag> condanna </hashtag> <hashtag> ingiustizia </hashtag> <hashtag> vergogna </hashtag> <hashtag> umili <org> azione </org> </hashtag> <hashtag> GovernodellaVergogna </hashtag> <hashtag> pfizer </hashtag> <hashtag> moderna </hashtag> <hashtag> sputnik </hashtag> <hashtag> JohnsonandJohnson </hashtag> <hashtag> senzavergogna </hashtag>,0,2363.0,0.0,0.400788,0.0,1,0.976045,False
434,231114690993031,<hashtag> <per> zaia </per> </hashtag> hai avuto il culo di trovare <hashtag> cri <per> santi </per> </hashtag> all'inizio. Ma poi l'arroganza tipica della <hashtag> <org> lega </org> </hashtag> ha avuto il sopravvento e ti sei incartato .. <hashtag> reportrai 3 </hashtag>,0,101.0,0.0,-1.091682,0.0,1,0.975946,False
79,292725196549878,".Anche oggi fermati 50 <hashtag> clandestini </hashtag> in zona Bagnoli della Rosandra(TS). Un ex poliziotto che vive nei pressi:""MAI visti pattugliamenti nei boschi, passa chiunque""<hashtag> TGR </hashtag> <hashtag> FVG </hashtag> I <hashtag> portichiusi </hashtag> di <hashtag> <per> Salvini </per> </hashtag> sono INUTILI! <hashtag> immigrati </hashtag> <hashtag> <per> Rotta </per> Bal <per> cani </pe


TEST
False negatives: 72
False positives: 47

False negatives — most confident mistakes first:


,anonymized_tweet_id,text,labels,followers_count,is_reply,followers_feature,reply_feature,prediction,probability_hate,correct
356,84399138297621,"<hashtag> parlatecidibibbiano </hashtag> oggi, <hashtag> <org> lega </org> ioli </hashtag>... <hashtag> Risultati </hashtag> <hashtag> ElezioniEmiliaRomagna </hashtag> <hashtag> EmiliaRomagna </hashtag> <hashtag> attrippato </hashtag> <hashtag> <per> bonaccini </per> presidente </hashtag> <hashtag> <per> Bonaccini </per> </hashtag> <hashtag> <per> Borgonzoni </per> </hashtag> <hashtag> <org> m5s </org> </hashtag>",1,74906,0,2.041765,0.0,0,0.002527,False
257,752007993450933,"Ieri in Senato si discuteva della questione <hashtag> Ilva </hashtag>, il senatore <hashtag> <per> salvini </per> </hashtag> era assente in quanto a <hashtag> Verona </hashtag> per la <hashtag> FieraCa <per> valli </per> </hashtag>. C'è un padiglione dedicato agli asini.",1,3875,0,0.635569,0.0,0,0.002742,False
239,300337469514409,"E dopo il <hashtag> redditodicittadinanza </hashtag>, <hashtag> Quota 100 </hashtag>, <hashtag> leggeanticorruzione </hashtag>, <hashtag> taglio <per> vitali </per> zi </hashtag>, <hashtag> decretodignità </hashtag>, quei ""poltronari"" del <hashtag> <org> M5S </org> </hashtag> riescono a far approvare anche il <hashtag> TAGLIAPOLTRONE </hashtag>.",1,4629,0,0.719973,0.0,0,0.003322,False
533,66355036290705,Sbaglio o la <hashtag> regolarizz <org> azione </org> </hashtag> dei <hashtag> mi <per> grant </per> i </hashtag> è l'esatto contrario del <hashtag> decre <per> tosi </per> curezza </hashtag> votato dagli stessi <hashtag> pentacretini </hashtag>?,1,4486,0,0.705076,0.0,0,0.003330,False
855,730420754782401,<hashtag> <per> Grillo </per> </hashtag> Non vi distraete. Questa settimana si propone: Corso accelerato per vivere senza fare un ca@@zo e ricevere quasi 14.000 € al mese. Il corso si tiene a <hashtag> montecitorio </hashtag> da certificato docente leghista iscrizioni gratuite.,1,12,0,-2.069853,0.0,0,0.003628,False
102,588516232616131,<hashtag> <org> pd </org> </hashtag> <hashtag> sx </hashtag> e <hashtag> <org> m5s </org> </hashtag> stanno lavorando per la nostra sostituzione etnica con gli africani. Il 26 maggio i sovranisti vinceranno per legittima difesa! <hashtag> voto <org> Lega </org> </hashtag>,1,12553,0,1.193612,0.0,0,0.003718,False
822,831513741092172,Vediamo in quanti firmano per mandare te a lavorare dopo 25 anni di nullafacenza? <hashtag> petizione <per> meloni </per> a <per> casa </per> </hashtag> <hashtag> <per> giorgiameloni </per> </hashtag> <hashtag> petizione </hashtag> <hashtag> politici </hashtag>,1,301,1,-0.576271,1.0,0,0.003826,False
96,246818227160136,"Le scuole crollano, mancano gli insegnanti, in molte parti di Italia non è garantito il tempo pieno e le famiglie devono conciliare lavoro e cura. E lui, il Ministro di qualunque cosa, a che pensa? Al grembiule. Al grembiule! Ridicolo. <hashtag> grembiule </hashtag> <hashtag> ministrodelgrembiule </hashtag> <hashtag> <per> salvini </per> </hashtag>",1,51469,0,1.863584,0.0,0,0.004623,False
751,953489123320103,"Domanda, così, per sapere. Credete sia possibile che <hashtag> <per> Formigoni </per> </hashtag> si soffochi con il suo vitalizio?",1,727,0,-0.158477,0.0,0,0.005267,False
235,933337615059895,"Da radio Capital. ""Sulla manovra confermato il deficit al 2,2% per il 2020"". A Bruxelles, sui media italiani tutti zitti. Un anno fa hanno frantumato i c*glioni per il 2,04%. Ha letto male lo speaker oppure lo stato si è rimesso a 90°? <hashtag> deficit </hashtag> <hashtag> <per> Conte </per> </hashtag> <hashtag> <org> m5s </org> pd </hashtag> <hashtag> giallorosso </hashtag>",1,148,0,-0.911733,0.0,0,0.005356,False



False positives — most confident mistakes first:


,anonymized_tweet_id,text,labels,followers_count,is_reply,followers_feature,reply_feature,prediction,probability_hate,correct
938,984345832091039,<hashtag> LaTtualità </hashtag> <hashtag> <per> Draghi </per> </hashtag> <hashtag> Governo <per> Draghi </per> </hashtag> <hashtag> <per> Salvini </per> <per> Paglia </per> ccio </hashtag> <hashtag> <per> Salvini </per> </hashtag> <hashtag> riaperture </hashtag> <hashtag> DecretoRiaperture </hashtag> <hashtag> DecretoCovid </hashtag> <hashtag> Coprifuoco </hashtag> aprire la foto!,0,9,0,-2.194432,0.0,1,0.997997,False
572,757948338056776,"A tutti i rancorosi che rilanciano <hashtag> <per> salvini </per> CIALTRONE </hashtag>, ma che razza di vita frustrante hanno per vomitare tanto livore, poi sopratutto contro un'esponente di minoranza. Chi è ossessionato dai <hashtag> 49 milioni </hashtag> di , si informino dei 81,6 milioni dell'Unità che pagherà.",0,366,0,-0.483709,0.0,1,0.994652,False
336,484408381542760,"<hashtag> dincà </hashtag> ha ragione...la gente non è stupida, ma nel capire il gioco delle tre carte del <hashtag> pd <org> m5s </org> </hashtag> Buffoni! Altroché <hashtag> <per> salvini </per> </hashtag> vittima! <hashtag> tagadala 7 </hashtag>",0,2190,0,0.364702,0.0,1,0.994457,False
1073,786885721910518,"Ma si, continuiamo a non curare i malati... <hashtag> <per> Speranza </per> Vattene </hashtag>",0,185,0,-0.806414,0.0,1,0.993548,False
604,591415938523671,"Cari signori vi segnalo che gli italiani non sono <hashtag> coglioni </hashtag>. Poteva far dimettere la <hashtag> <per> riccardi </per> </hashtag> e quell altro come fece il <hashtag> <org> m5s </org> </hashtag>. Ripeto, non siamo Coglioni",0,40,1,-1.524447,1.0,1,0.993410,False
508,917719852951076,"<hashtag> Report </hashtag> <hashtag> RenzoRosso </hashtag> <hashtag> Quartarepubblica </hashtag> Mascherine prezzo fissato 0,50 Il Manzoni scrisse' che durante la peste ci fu il prezzo fissato del PANE, ma siccome a quel prezzo nessuno lo vendeva ci fu una rivoluzione e l' assalto ai forni SONO COSI' IGNORANTI questi del <hashtag> <per> conte </per> dimettiti </hashtag>",0,2479,0,0.423534,0.0,1,0.992957,False
1385,657810552265107,Il ducetto <hashtag> <per> Letta </per> </hashtag> sta vaneggiando? Ma cosa dice? <hashtag> èfuoridallarealtà </hashtag> <hashtag> borioso </hashtag> <hashtag> supponente </hashtag> 😛 😛 😛,0,32,1,-1.627517,1.0,1,0.989318,False
715,717701472339912,"Non mi permetto di giudicare il presidente <hashtag> <per> Mattarella </per> </hashtag> circa il suo silenzio sull'ingiustizia sociale dei vitalizi ripristinati, penso che, se fossimo un popolo serio, scenderemmo in piazza per fare in modo che questa grande schifezza abbia fine <hashtag> <per> vitali </per> zi </hashtag> <hashtag> ladri </hashtag>",0,435,0,-0.401905,0.0,1,0.987183,False
290,970396202587744,"la faccia acida della <hashtag> meli </hashtag> che sprizza <hashtag> odio </hashtag> sputacchiando domande preparate a tavolino, la <hashtag> berlinguer </hashtag> che agita le mani sudaticcie non hanno prezzo <hashtag> <per> salvini </per> nonmollare </hashtag> <hashtag> cartabianca </hashtag>",0,452,0,-0.383742,0.0,1,0.982505,False
738,352866062034382,"Nei confronti di <hashtag> <per> Formigoni </per> </hashtag> nessuna vergogna? Avete votato a favore,lo schifo proprio",0,1055,1,0.018135,1.0,1,0.976987,False



Saved CSV files to: /content/drive/MyDrive/master_unipa/alberto_wiki_error_analysis_final
